# Pediatric Bone Age Estimation — Iteration 1

**Dataset:** RSNA Pediatric Bone Age (12,611 left-hand radiographs, bone age in months, sex)

This notebook rebuilds the course submission (`Submission/group_36_new.ipynb`, dual-stream Xception + ResNet50, reported **R² = 0.934, MAE = 7.95 mo**) into an end-to-end, reproducible pipeline and adds three ideas built specifically around how bone age is assessed:

| # | Contribution | Why it fits this problem |
|---|---|---|
| 1 | **Topological input stream** — cubical persistent homology of the radiograph (H₀/H₁ under a superlevel-set filtration), vectorised into Betti / lifespan curves and fused with the CNN | Skeletal maturation is largely a *topological* process: carpal ossification centres **appear** (new connected components) and epiphyses **fuse** with metaphyses (components merge, the radiolucent physis disappears). Persistence summaries are also invariant to hand rotation/translation — a known failure mode of the submission. |
| 2 | **Atlas-Aware Distribution Loss (AADL)** — predict a distribution over ages; the soft target's width follows the spacing of Greulich–Pyle standards, estimated from the label lattice per sex | RSNA labels are GP-atlas readings, not continuous measurements: a label of 132 mo means "closest standard is 132 mo". Label uncertainty is therefore proportional to the local spacing of the atlas, which varies with age and sex. The loss encodes that, and the predicted spread gives uncertainty for free. |
| 3 | **Principled post-processing** — fold-ensemble + flip TTA, sex-specific regression-to-the-mean calibration fitted on out-of-fold predictions only, and split-conformal 90 % prediction intervals | Regression models shrink extremes toward the mean, which hurts R² most at the youngest/oldest ages. Calibration fixes this without touching the test set; conformal intervals turn the output into something clinically reportable. |

Plus the engineering fixes that move R² the most: aspect-preserving hand crop + CLAHE, augmentation, a modern backbone at 512 px, AdamW + cosine + EMA, and a **held-out test set never used for model selection**.

**Target.** With label SD ≈ 42 mo, R² = 0.97 ⇔ RMSE ≈ 7.3 mo and R² = 0.98 ⇔ RMSE ≈ 5.9 mo (computed exactly below).

> **Compute.** Defaults target a single CUDA GPU (Kaggle T4/P100 or Colab). Every run is checkpointed and skipped on re-execution, so the notebook can be resumed across sessions. Set `BA_QUICK=1` for a 2-minute end-to-end dry run.

## 0 · What changed vs. the submission

| Issue in submission | Fix here |
|---|---|
| Training code was inside a string literal (`Config.LR` undefined) — results not reproducible | Full training engine in-notebook, resumable, all configs logged |
| Checkpoint selected on the same split used to report R² | Stratified **85 / 15 dev / test**; 5-fold CV inside dev; test touched only for final reporting |
| No augmentation; image squashed to 512² then 299² (two aspect distortions) | Rotation/scale/shift/flip/contrast/erasing; aspect-preserving letterbox |
| Fixed-percentage crops assume an upright, frame-filling hand | Automatic hand localisation (Otsu + connected components) + CLAHE |
| L1 on raw months; sex as a 32-d concat | Standardised targets; distribution head; **FiLM** sex conditioning |
| Inconsistent stage bins (0/3/10/15 vs 0/5/10/15) | One definition (0–3 / 3–10 / 10–15 / 15+ y), fixed label order |
| SSL verification disabled globally | Removed |

## 1 · Setup

In [ ]:
%pip install -q timm gudhi

In [ ]:
import os, gc, json, math, time, copy, random, platform, warnings, contextlib
from dataclasses import dataclass, field, replace, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as T
import timm
import gudhi
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score,
                             f1_score, cohen_kappa_score, confusion_matrix)
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=UserWarning)
cv2.setNumThreads(0)
sns.set_theme(style="whitegrid", context="notebook")
SEX_COLORS = {0: "#d1495b", 1: "#2e86ab"}   # 0 = female, 1 = male
SEX_NAMES = {0: "Female", 1: "Male"}
print(f"torch {torch.__version__} | timm {timm.__version__} | gudhi {gudhi.__version__} | cv2 {cv2.__version__}")

## 2 · Configuration
Every hyper-parameter lives here. Any field can be overridden from the environment as `BA_<FIELD>` (e.g. `BA_IMG_SIZE=384`), and each experiment in §8 is just a set of overrides on this object.

In [ ]:
def _find_data_dir():
    candidates = [os.environ.get("BONEAGE_DATA_DIR"), "/kaggle/input/rsna-bone-age",
                  "/content/rsna-bone-age", "./data/rsna-bone-age", "../data/rsna-bone-age"]
    for c in candidates:
        if c and Path(c, "boneage-training-dataset.csv").exists():
            return Path(c)
    if Path("/kaggle/input").exists():
        for p in Path("/kaggle/input").rglob("boneage-training-dataset.csv"):
            return p.parent
    raise FileNotFoundError("RSNA Bone Age data not found - set BONEAGE_DATA_DIR to the folder "
                            "containing boneage-training-dataset.csv")

def _default_work_dir():
    return Path("/kaggle/working/iteration_1") if Path("/kaggle/working").exists() else Path("runs/iteration_1")

def _device():
    if torch.cuda.is_available(): return "cuda"
    if torch.backends.mps.is_available(): return "mps"
    return "cpu"

@dataclass
class Config:
    DATA_DIR: Path = field(default_factory=_find_data_dir)
    WORK_DIR: Path = field(default_factory=_default_work_dir)
    QUICK: bool = False               # tiny end-to-end dry run
    MAX_IMAGES: int = 0               # >0: subsample the dataset (debugging)
    SEED: int = 42
    # data
    IMG_SIZE: int = 512
    TEST_FRAC: float = 0.15
    N_FOLDS: int = 5
    # model / optimisation
    BACKBONE: str = "convnext_tiny.fb_in22k_ft_in1k"
    PRETRAINED: bool = True
    DROP_PATH: float = 0.1
    EPOCHS: int = 25
    BATCH_SIZE: int = 16
    LR_BACKBONE: float = 1e-4
    LR_HEAD: float = 1e-3
    WEIGHT_DECAY: float = 0.05
    WARMUP_EPOCHS: float = 1.0
    EMA_DECAY: float = 0.998
    PATIENCE: int = 8
    # stop cleanly (after checkpointing) once the session has run this long; 0 = off. Kaggle sessions cap at 12 h.
    TIME_BUDGET_MIN: float = field(default_factory=lambda: 690.0 if Path("/kaggle/working").exists() else 0.0)
    # loss: "l1" (submission baseline) | "laplace" (heteroscedastic NLL) | "aadl" (ours)
    LOSS: str = "aadl"
    MAX_AGE: int = 240                # months; distribution head support 0..MAX_AGE
    ATLAS_MIN_FREQ: float = 0.004     # a label value is an "atlas standard" if >= 0.4 % of that sex's labels
    AADL_GAMMA: float = 0.5           # sigma = gamma * local atlas spacing
    AADL_SIGMA_MIN: float = 2.0
    AADL_SIGMA_MAX: float = 18.0
    AADL_SIGMA_DEFAULT: float = 6.0   # used if no lattice is detected
    AADL_LAMBDA: float = 1.0          # weight of the expectation-L1 term (years)
    # topological stream
    USE_TDA: bool = True
    TDA_RES: int = 32                 # thresholds per Betti / lifespan curve
    TDA_MIN_PERS: float = 0.02
    TDA_SCALES: tuple = ((128, 1.0), (256, 1.5))   # (width px, gaussian sigma) for the whole hand
    TDA_BANDS: bool = True            # also fingers / metacarpals / wrist bands at the largest scale
    # inference
    TTA: bool = True
    CONFORMAL_ALPHA: float = 0.10
    NUM_WORKERS: int = field(default_factory=lambda: min(4, os.cpu_count() or 1) if platform.system() == "Linux" else 0)
    DEVICE: str = field(default_factory=_device)

def _apply_env(cfg):
    upd = {}
    for k, v in asdict(cfg).items():
        env = os.environ.get(f"BA_{k}")
        if env is None or isinstance(v, tuple):
            continue
        if isinstance(v, bool):    upd[k] = env.lower() in ("1", "true", "yes")
        elif isinstance(v, int):   upd[k] = int(env)
        elif isinstance(v, float): upd[k] = float(env)
        elif isinstance(v, Path):  upd[k] = Path(env)
        else:                      upd[k] = env
    return replace(cfg, **upd)

CFG = _apply_env(Config())
if CFG.QUICK:
    CFG = replace(CFG, IMG_SIZE=128, EPOCHS=2, N_FOLDS=2, BATCH_SIZE=8, BACKBONE="resnet18",
                  MAX_IMAGES=CFG.MAX_IMAGES or 240, TDA_SCALES=((64, 1.0), (96, 1.0)),
                  WORK_DIR=Path(str(CFG.WORK_DIR) + "_quick"))
CFG.WORK_DIR.mkdir(parents=True, exist_ok=True)
(CFG.WORK_DIR / "figures").mkdir(exist_ok=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True

SESSION_T0 = time.time()
seed_everything(CFG.SEED)
print(f"device={CFG.DEVICE} | data={CFG.DATA_DIR} | work={CFG.WORK_DIR} | quick={CFG.QUICK}")

## 3 · Data and the label lattice

In [ ]:
def load_metadata(cfg):
    df = pd.read_csv(cfg.DATA_DIR / "boneage-training-dataset.csv")
    df["male"] = df["male"].astype(str).str.lower().isin(["true", "1"]).astype(int)
    paths = {p.stem: p for p in cfg.DATA_DIR.rglob("*.png") if "training" in str(p.parent).lower()}
    df["path"] = df["id"].astype(str).map(paths)
    n_missing = int(df["path"].isna().sum())
    df = df.dropna(subset=["path"])
    if cfg.MAX_IMAGES:
        df = df.sample(n=min(cfg.MAX_IMAGES, len(df)), random_state=cfg.SEED)
    df = df.sort_values("id").reset_index(drop=True)
    print(f"{len(df):,} images ({n_missing} listed in CSV but missing on disk)")
    return df

df = load_metadata(CFG)
summary = df.groupby("male")["boneage"].agg(["count", "mean", "std", "min", "max"]).rename(index=SEX_NAMES)
display(summary.round(1))

Y_SD = df["boneage"].std()
display(pd.DataFrame({"R²": [0.934, 0.95, 0.97, 0.98]}).assign(
    **{"required RMSE (mo)": lambda d: Y_SD * np.sqrt(1 - d["R²"])}).round(2).set_index("R²").T)

RSNA labels are radiologist readings against the Greulich–Pyle atlas, so they pile up on a small set of *standard* ages. The right panel shows that lattice directly; it is the motivation for the atlas-aware loss in §6.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.2))
for s in (0, 1):
    sns.histplot(df.loc[df.male == s, "boneage"] / 12, bins=40, ax=ax[0], color=SEX_COLORS[s],
                 label=SEX_NAMES[s], alpha=0.55, stat="density")
ax[0].set(xlabel="Bone age (years)", title="Age distribution by sex"); ax[0].legend()
for s, off in ((0, -0.3), (1, 0.3)):
    vc = df.loc[df.male == s, "boneage"].round().value_counts(normalize=True).sort_index()
    ax[1].vlines(vc.index + off, 0, vc.values, color=SEX_COLORS[s], lw=1.6, label=SEX_NAMES[s])
ax[1].set(xlabel="Label value (months)", ylabel="Share of labels", title="Label lattice — mass sits on atlas standards")
ax[1].legend(); plt.tight_layout(); plt.savefig(CFG.WORK_DIR / "figures/eda_labels.png", dpi=150); plt.show()

for s in (0, 1):
    vc = df.loc[df.male == s, "boneage"].round().value_counts(normalize=True)
    on = vc[vc >= CFG.ATLAS_MIN_FREQ]
    print(f"{SEX_NAMES[s]:>6}: {len(on)} label values carry {on.sum():.1%} of all labels")

## 4 · Leakage-free split
85 % *development* / 15 % *test*, stratified by sex × age decile. The development set is split into 5 folds; each fold's validation part drives early stopping and calibration. **The test set is used only in §10.**

In [ ]:
def make_splits(df, cfg):
    df = df.copy()
    strat = df["male"].astype(str) + "_" + pd.qcut(df["boneage"], 10, labels=False, duplicates="drop").astype(str)
    dev_idx, test_idx = train_test_split(np.arange(len(df)), test_size=cfg.TEST_FRAC,
                                         stratify=strat, random_state=cfg.SEED)
    df["split"] = "dev"
    df.loc[test_idx, "split"] = "test"
    df["fold"] = -1
    skf = StratifiedKFold(cfg.N_FOLDS, shuffle=True, random_state=cfg.SEED)
    for f, (_, vi) in enumerate(skf.split(dev_idx, strat.iloc[dev_idx])):
        df.loc[dev_idx[vi], "fold"] = f
    return df

df = make_splits(df, CFG)
df.drop(columns=["path"]).to_csv(CFG.WORK_DIR / "splits.csv", index=False)
display(pd.crosstab(df["split"], df["fold"], margins=True))

## 5 · Preprocessing
1. Robust intensity normalisation (0.5–99.5 percentile) — handles 8/16-bit PNGs and RGB exports.
2. Inverted-image check (bright background ⇒ invert).
3. **Hand localisation:** Otsu on a blurred thumbnail (threshold halved so soft tissue is included), keep the largest component plus any comparably large ones, take the union box + 4 % margin. Falls back to the full frame if the mask looks implausible.
4. Resize the longest side to `IMG_SIZE` → CLAHE → **letterbox** to a square (no aspect distortion).

Preprocessed images are cached to disk once, which also removes the per-batch PNG decoding of 2000-px originals that made the submission's inference take 2 s / batch.

In [ ]:
def read_xray(path):
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise IOError(f"cannot read {path}")
    if img.ndim == 3:
        img = cv2.cvtColor(img[..., :3], cv2.COLOR_BGR2GRAY)
    img = img.astype(np.float32)
    lo, hi = np.percentile(img, (0.5, 99.5))
    img = np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1)
    return (img * 255).astype(np.uint8)

def is_inverted(img8):
    h, w = img8.shape
    b = max(2, int(0.03 * min(h, w)))
    border = np.concatenate([img8[:b].ravel(), img8[-b:].ravel(), img8[:, :b].ravel(), img8[:, -b:].ravel()])
    return np.median(border) > 200

def hand_bbox(img8):
    s = min(1.0, 512 / max(img8.shape))
    small = cv2.resize(img8, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else img8
    blur = cv2.GaussianBlur(small, (0, 0), 3)
    otsu, _ = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = (blur > 0.5 * otsu).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n <= 1:
        return None
    areas = stats[1:, cv2.CC_STAT_AREA]
    keep = 1 + np.flatnonzero(areas >= 0.25 * areas.max())
    x0 = stats[keep, cv2.CC_STAT_LEFT].min(); y0 = stats[keep, cv2.CC_STAT_TOP].min()
    x1 = (stats[keep, cv2.CC_STAT_LEFT] + stats[keep, cv2.CC_STAT_WIDTH]).max()
    y1 = (stats[keep, cv2.CC_STAT_TOP] + stats[keep, cv2.CC_STAT_HEIGHT]).max()
    box_frac = (x1 - x0) * (y1 - y0) / mask.size
    if box_frac < 0.10:          # implausibly small -> keep full frame
        return None
    return np.array([x0, y0, x1, y1]) / s

def preprocess(path, size, clahe_clip=2.0):
    img = read_xray(path)
    if is_inverted(img):
        img = 255 - img
    box = hand_bbox(img)
    if box is not None:
        H, W = img.shape
        x0, y0, x1, y1 = box
        m = 0.04 * max(x1 - x0, y1 - y0)
        img = img[int(max(0, y0 - m)):int(min(H, y1 + m)), int(max(0, x0 - m)):int(min(W, x1 + m))]
    h, w = img.shape
    s = size / max(h, w)
    img = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))), interpolation=cv2.INTER_AREA)
    img = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=(8, 8)).apply(img)
    canvas = np.zeros((size, size), np.uint8)
    h, w = img.shape
    y0, x0 = (size - h) // 2, (size - w) // 2
    canvas[y0:y0 + h, x0:x0 + w] = img
    return canvas

def build_cache(df, cfg):
    cache = cfg.WORK_DIR / f"cache_{cfg.IMG_SIZE}"
    cache.mkdir(exist_ok=True)
    todo = [(i, p) for i, p in zip(df["id"], df["path"]) if not (cache / f"{i}.png").exists()]
    def work(i, p):
        cv2.imwrite(str(cache / f"{i}.png"), preprocess(p, cfg.IMG_SIZE))
    if todo:
        Parallel(n_jobs=-1, prefer="threads")(delayed(work)(i, p) for i, p in tqdm(todo, desc="preprocess"))
    return [str(cache / f"{i}.png") for i in df["id"]]

df["cache_path"] = build_cache(df, CFG)

In [ ]:
ex = df.sample(6, random_state=1)
fig, ax = plt.subplots(2, 6, figsize=(18, 6.5))
for j, r in enumerate(ex.itertuples()):
    ax[0, j].imshow(read_xray(r.path), cmap="gray"); ax[0, j].set_title(f"raw · {r.boneage/12:.1f} y")
    ax[1, j].imshow(cv2.imread(r.cache_path, cv2.IMREAD_GRAYSCALE), cmap="gray"); ax[1, j].set_title("crop + CLAHE + letterbox")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(CFG.WORK_DIR / "figures/preprocessing.png", dpi=120); plt.show()

## 6 · Novel input stream — topology of ossification

Treat the radiograph as a function $f:\Omega\to[0,1]$ and sweep a threshold $t$ from bright to dark. The superlevel sets $\{f \ge t\}$ form a filtration whose **persistent homology** records when bright structures are *born* and when they *merge*:

* **H₀ (connected components).** Each ossification centre is born as its own component. An *open* growth plate is a dark gap between epiphysis and metaphysis, so the two are separate components that only merge at the gap intensity — persistence = epiphysis brightness − physis brightness. As the plate closes this persistence → 0. Carpal centres appear in a known order (capitate/hamate → triquetrum → lunate → trapezium/trapezoid/scaphoid → pisiform), adding components with age.
* **H₁ (loops).** Cortical rings and inter-bone spaces enclosed by bone.

Each diagram is vectorised as a **Betti curve** β(t), a **lifespan curve** (summed persistence of features alive at t) and nine summary statistics, computed on the whole hand at two scales and on three anatomical bands (fingers / metacarpals / wrist). These features are invariant to in-plane rotation and translation of the hand, they are computed once, and they enter the network through a small MLP (§7).

In [ ]:
def superlevel_diagrams(im, min_pers):
    """Persistence diagrams of the superlevel filtration of im (values in [0,1]).
    Returns {dim: array[n, 2]} with columns (birth intensity, death intensity), birth > death."""
    cc = gudhi.CubicalComplex(top_dimensional_cells=-im)
    cc.compute_persistence(homology_coeff_field=2, min_persistence=min_pers)
    out = {}
    for dim in (0, 1):
        iv = np.asarray(cc.persistence_intervals_in_dimension(dim), dtype=np.float64).reshape(-1, 2)
        birth, death = -iv[:, 0], -iv[:, 1]
        death[~np.isfinite(death)] = 0.0              # essential class lives until the darkest level
        out[dim] = np.stack([np.clip(birth, 0, 1), np.clip(death, 0, 1)], 1)
    return out

def vectorize_diagram(bd, grid):
    birth, death = bd[:, 0], bd[:, 1]
    pers = birth - death
    alive = (grid[None, :] <= birth[:, None]) & (grid[None, :] > death[:, None])
    betti = alive.sum(0)
    life = (alive * pers[:, None]).sum(0)
    if len(pers) and pers.sum() > 0:
        p = pers / pers.sum()
        stats = [len(pers), pers.sum(), pers.max(), pers.mean(), pers.std(), -(p * np.log(p + 1e-12)).sum(),
                 (pers > 0.05).sum(), (pers > 0.1).sum(), (pers > 0.2).sum()]
    else:
        stats = [0.0] * 9
    return np.concatenate([betti, life, stats]).astype(np.float32)

TDA_BANDS = (("fingers", 0.0, 0.4), ("metacarpals", 0.3, 0.7), ("wrist", 0.6, 1.0))

def tda_regions(img, scales, bands):
    """(name, image, width, sigma) for every region/scale the features are computed on."""
    regs = [(f"whole{w}", img, w, s) for w, s in scales]
    if bands:
        H = img.shape[0]
        w, s = scales[-1]
        regs += [(name, img[int(a * H):int(b * H)], w, s) for name, a, b in bands]
    return regs

def tda_features(path, res, min_pers, scales, bands):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
    grid = np.linspace(0, 1, res)
    feats = []
    for _, reg, width, sigma in tda_regions(img, scales, bands if bands else ()):
        h, w = reg.shape
        reg = cv2.resize(reg, (width, max(8, round(h * width / w))), interpolation=cv2.INTER_AREA)
        reg = cv2.GaussianBlur(reg, (0, 0), sigma)
        dg = superlevel_diagrams(reg, min_pers)
        feats += [vectorize_diagram(dg[0], grid), vectorize_diagram(dg[1], grid)]
    return np.concatenate(feats)

def tda_feature_names(cfg):
    stats = ["n", "total", "max", "mean", "std", "entropy", "n>0.05", "n>0.1", "n>0.2"]
    names = []
    for reg, *_ in tda_regions(np.zeros((10, 10)), cfg.TDA_SCALES, TDA_BANDS if cfg.TDA_BANDS else ()):
        for h in ("H0", "H1"):
            names += [f"{reg}|{h}|betti{k}" for k in range(cfg.TDA_RES)]
            names += [f"{reg}|{h}|life{k}" for k in range(cfg.TDA_RES)]
            names += [f"{reg}|{h}|{s}" for s in stats]
    return names

def build_tda(df, cfg):
    f = cfg.WORK_DIR / f"tda_{cfg.IMG_SIZE}_{cfg.TDA_RES}_{len(cfg.TDA_SCALES)}{int(cfg.TDA_BANDS)}.npz"
    known = {}
    if f.exists():
        z = np.load(f)
        known = dict(zip(z["ids"].tolist(), z["X"]))
    todo = [(i, p) for i, p in zip(df["id"], df["cache_path"]) if i not in known]
    if todo:
        args = (cfg.TDA_RES, cfg.TDA_MIN_PERS, cfg.TDA_SCALES, TDA_BANDS if cfg.TDA_BANDS else ())
        X = Parallel(n_jobs=-1, batch_size=16)(delayed(tda_features)(p, *args) for _, p in tqdm(todo, desc="persistent homology"))
        known.update({i: x for (i, _), x in zip(todo, X)})
        np.savez_compressed(f, ids=np.array(list(known)), X=np.stack(list(known.values())))
    return np.stack([known[i] for i in df["id"]]).astype(np.float32)

t0 = time.time()
TDA_X = build_tda(df, CFG)
TDA_NAMES = tda_feature_names(CFG)
assert TDA_X.shape[1] == len(TDA_NAMES)
print(f"TDA features: {TDA_X.shape} in {time.time() - t0:.0f}s")

**What the topology sees.** Three hands of the same sex at increasing age: the H₀ diagram (each dot is one bright structure; distance below the diagonal = persistence), and the Betti-0 curves.

In [ ]:
sex = int(df["male"].mode()[0])
pool = df[df.male == sex]
picks = [pool.iloc[(pool.boneage - a).abs().argmin()] for a in (24, 96, 180)]
fig, ax = plt.subplots(1, 5, figsize=(21, 4.2), gridspec_kw={"width_ratios": [1, 1, 1, 1.25, 1.25]})
grid = np.linspace(0, 1, CFG.TDA_RES)
w, sig = CFG.TDA_SCALES[-1]
cols = plt.cm.viridis(np.linspace(0.1, 0.85, 3))
for j, r in enumerate(picks):
    img = cv2.imread(r.cache_path, cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255
    ax[j].imshow(img, cmap="gray"); ax[j].axis("off"); ax[j].set_title(f"{r.boneage/12:.1f} y")
    im = cv2.GaussianBlur(cv2.resize(img, (w, w), interpolation=cv2.INTER_AREA), (0, 0), sig)
    dg = superlevel_diagrams(im, CFG.TDA_MIN_PERS)
    ax[3].scatter(dg[0][:, 0], dg[0][:, 1], s=10, alpha=0.6, color=cols[j], label=f"{r.boneage/12:.1f} y")
    ax[4].plot(grid, vectorize_diagram(dg[0], grid)[:CFG.TDA_RES], color=cols[j], lw=2, label=f"{r.boneage/12:.1f} y")
ax[3].plot([0, 1], [0, 1], "k--", lw=1)
ax[3].set(xlabel="birth intensity", ylabel="death intensity", title="$H_0$ persistence diagram"); ax[3].legend()
ax[4].set(xlabel="intensity threshold t", ylabel=r"$\beta_0(t)$", title="Betti-0 curve"); ax[4].legend()
plt.tight_layout(); plt.savefig(CFG.WORK_DIR / "figures/tda_examples.png", dpi=130); plt.show()

**Is there signal on its own?** Spearman correlation of each topological feature with bone age, and a gradient-boosting model trained on *topology + sex only* (no CNN) and evaluated on the held-out test set.

In [ ]:
dev_mask = (df.split == "dev").values
tda_df = pd.DataFrame(np.log1p(TDA_X), columns=TDA_NAMES)
rho = tda_df[dev_mask].corrwith(df.loc[dev_mask, "boneage"], method="spearman").dropna()
top = rho.reindex(rho.abs().sort_values(ascending=False).index)[:20]

hgb = HistGradientBoostingRegressor(max_iter=600, learning_rate=0.05, max_leaf_nodes=31, random_state=CFG.SEED)
Xg = np.c_[np.log1p(TDA_X), df["male"].values]
hgb.fit(Xg[dev_mask], df.loc[dev_mask, "boneage"])
p_tda = hgb.predict(Xg[~dev_mask]); y_te = df.loc[~dev_mask, "boneage"].values
TDA_ONLY = {"MAE_mo": mean_absolute_error(y_te, p_tda), "R2": r2_score(y_te, p_tda)}

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].barh(top.index[::-1], top.values[::-1], color=["#2e86ab" if v > 0 else "#d1495b" for v in top.values[::-1]])
ax[0].set(xlabel="Spearman ρ with bone age", title="Most age-informative topological features")
ax[1].scatter(y_te / 12, p_tda / 12, s=8, alpha=0.4, color="#555")
ax[1].plot([0, 20], [0, 20], "k--", lw=1)
ax[1].set(xlabel="True (y)", ylabel="Predicted (y)",
          title=f"Topology-only model (test)  MAE {TDA_ONLY['MAE_mo']:.1f} mo · R² {TDA_ONLY['R2']:.3f}")
plt.tight_layout(); plt.savefig(CFG.WORK_DIR / "figures/tda_signal.png", dpi=130); plt.show()

## 7 · Atlas-Aware Distribution Loss (AADL)

The head outputs logits over integer ages $c_k \in \{0,\dots,240\}$ months, giving a distribution $p$. For a label $y$ of sex $s$ the target is a discretised Gaussian

$$q_k \propto \exp\!\Big(-\frac{(c_k-y)^2}{2\,\sigma_s(y)^2}\Big), \qquad \sigma_s(y)=\operatorname{clip}\big(\gamma\cdot\Delta_s(y),\ \sigma_{\min},\ \sigma_{\max}\big)$$

where $\Delta_s(y)$ is the local spacing between consecutive **atlas standards** for sex $s$, estimated from the training labels alone (values holding ≥ 0.4 % of that sex's labels). The loss is

$$\mathcal{L} = \underbrace{-\textstyle\sum_k q_k \log p_k}_{\text{distribution matching}} \;+\; \lambda\,\big|\,\mathbb{E}_p[c]-y\,\big| / 12 .$$

The prediction is $\hat y=\mathbb{E}_p[c]$ and the uncertainty is $\hat\sigma=\sqrt{\operatorname{Var}_p[c]}$. This extends label-distribution / mean-variance learning from facial age estimation (DLDL-v2, Gao et al. 2018; Pan et al. 2018) with a noise model that comes from the atlas the labels were read against.

Baselines in the ablation: **L1** (as in the submission) and **heteroscedastic Laplace NLL** (L1 with a learned per-image scale).

In [ ]:
def atlas_sigma_table(train_df, cfg):
    """sigma_s(m) for s in {0,1} and m = 0..MAX_AGE, from the label lattice of the training rows only."""
    m = np.arange(cfg.MAX_AGE + 1)
    table, standards = np.zeros((2, cfg.MAX_AGE + 1), np.float32), {}
    for s in (0, 1):
        freq = train_df.loc[train_df.male == s, "boneage"].round().astype(int).value_counts(normalize=True)
        std_vals = np.sort(freq[freq >= cfg.ATLAS_MIN_FREQ].index.values)
        standards[s] = std_vals
        if len(std_vals) < 8:
            table[s] = cfg.AADL_SIGMA_DEFAULT
            continue
        gaps = np.diff(std_vals).astype(float)
        spacing = np.r_[gaps[0], (gaps[:-1] + gaps[1:]) / 2, gaps[-1]]
        table[s] = np.clip(cfg.AADL_GAMMA * np.interp(m, std_vals, spacing), cfg.AADL_SIGMA_MIN, cfg.AADL_SIGMA_MAX)
    return table, standards

class Criterion:
    def __init__(self, cfg, sigma_table, y_mean, y_std):
        self.cfg, self.y_mean, self.y_std = cfg, float(y_mean), float(y_std)
        self.centers = torch.arange(cfg.MAX_AGE + 1, dtype=torch.float32)
        self.sigma = torch.as_tensor(sigma_table)

    def to(self, device):
        self.centers, self.sigma = self.centers.to(device), self.sigma.to(device)
        return self

    def decode(self, out):
        """head output -> (mean, std) in months; std is NaN for the plain-L1 model."""
        out = out.float()
        if self.cfg.LOSS == "aadl":
            p = out.softmax(1)
            mean = p @ self.centers
            var = (p @ self.centers ** 2 - mean ** 2).clamp_min(1e-6)
            return mean, var.sqrt()
        mean = out[:, 0] * self.y_std + self.y_mean
        if self.cfg.LOSS == "laplace":
            b = F.softplus(out[:, 1]) * self.y_std + 0.5
            return mean, math.sqrt(2) * b
        return mean, torch.full_like(mean, float("nan"))

    def __call__(self, out, y, sex):
        out = out.float()
        mean, sd = self.decode(out)
        if self.cfg.LOSS == "l1":
            return F.l1_loss(mean, y) / 12
        if self.cfg.LOSS == "laplace":
            b = sd / math.sqrt(2) / 12
            return (torch.log(2 * b) + (mean - y).abs() / 12 / b).mean()
        sig = self.sigma[sex, y.round().long().clamp(0, self.cfg.MAX_AGE)]
        q = torch.exp(-(self.centers[None] - y[:, None]) ** 2 / (2 * sig[:, None] ** 2))
        q = q / q.sum(1, keepdim=True)
        ce = -(q * out.log_softmax(1)).sum(1).mean()
        return ce + self.cfg.AADL_LAMBDA * (mean - y).abs().mean() / 12

sig_tab, standards = atlas_sigma_table(df[df.split == "dev"], CFG)
fig, ax = plt.subplots(figsize=(10, 3.6))
for s in (0, 1):
    ax.plot(np.arange(CFG.MAX_AGE + 1) / 12, sig_tab[s], color=SEX_COLORS[s], lw=2, label=f"{SEX_NAMES[s]} σ(y)")
    ax.plot(standards[s] / 12, np.interp(standards[s], np.arange(CFG.MAX_AGE + 1), sig_tab[s]), "o",
            color=SEX_COLORS[s], ms=3)
ax.set(xlabel="Bone age (years)", ylabel="target σ (months)", title="Atlas-aware label uncertainty (dots = detected standards)")
ax.legend(); plt.tight_layout(); plt.savefig(CFG.WORK_DIR / "figures/aadl_sigma.png", dpi=130); plt.show()

## 8 · Model

```
radiograph 1×512×512 ─► ConvNeXt-T (ImageNet-22k) ─► 768-d ─► FiLM(sex) ─┐
sex ─► embedding 32-d ────────────────────────────────────────────────────┼─► LayerNorm ─► MLP ─► head
persistent-homology vector ─► MLP ─► 128-d ───────────────────────────────┘
                                         head: 1 (L1) · 2 (Laplace) · 241-way distribution (AADL)
```
Sex enters twice: as an input and as a **FiLM** scale/shift on the image embedding (initialised to identity), because the same radiographic appearance maps to different ages in boys and girls. The backbone runs on one channel (timm sums the pretrained RGB filters).

In [ ]:
class BoneAgeNet(nn.Module):
    def __init__(self, cfg, n_tda):
        super().__init__()
        self.backbone = timm.create_model(cfg.BACKBONE, pretrained=cfg.PRETRAINED, num_classes=0,
                                          in_chans=1, drop_path_rate=cfg.DROP_PATH)
        d = self.backbone.num_features
        self.sex_emb = nn.Embedding(2, 32)
        self.film = nn.Linear(32, 2 * d)
        nn.init.zeros_(self.film.weight); nn.init.zeros_(self.film.bias)
        self.use_tda = cfg.USE_TDA and n_tda > 0
        t = 0
        if self.use_tda:
            self.tda_net = nn.Sequential(nn.Linear(n_tda, 256), nn.GELU(), nn.Dropout(0.2), nn.Linear(256, 128), nn.GELU())
            t = 128
        out_dim = {"l1": 1, "laplace": 2, "aadl": cfg.MAX_AGE + 1}[cfg.LOSS]
        fd = d + 32 + t
        self.head = nn.Sequential(nn.LayerNorm(fd), nn.Dropout(0.3), nn.Linear(fd, 512), nn.GELU(),
                                  nn.Dropout(0.1), nn.Linear(512, out_dim))

    def head_from_features(self, fmap, sex, tda):
        f = self.backbone.forward_head(fmap, pre_logits=True)
        s = self.sex_emb(sex)
        gamma, beta = self.film(s).chunk(2, dim=-1)
        parts = [f * (1 + gamma) + beta, s]
        if self.use_tda:
            parts.append(self.tda_net(tda))
        return self.head(torch.cat(parts, 1))

    def forward(self, x, sex, tda):
        return self.head_from_features(self.backbone.forward_features(x), sex, tda)


class BoneAgeDataset(Dataset):
    MEAN, STD = 0.449, 0.226

    def __init__(self, frame, tda=None, train=False):
        self.paths = frame["cache_path"].values
        self.sex = frame["male"].values.astype(np.int64)
        self.y = frame["boneage"].values.astype(np.float32)
        self.tda = tda
        self.train = train
        self.aug = T.Compose([
            T.RandomHorizontalFlip(),
            T.RandomAffine(degrees=20, translate=(0.06, 0.06), scale=(0.85, 1.15), shear=5),
            T.ColorJitter(brightness=0.2, contrast=0.2),
        ])
        self.erase = T.RandomErasing(p=0.25, scale=(0.01, 0.05), value=0)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        x = torch.from_numpy(cv2.imread(self.paths[i], cv2.IMREAD_GRAYSCALE))[None]
        if self.train:
            x = self.aug(x)
        x = (x.float() / 255 - self.MEAN) / self.STD
        if self.train:
            x = self.erase(x)
        t = torch.from_numpy(self.tda[i]) if self.tda is not None else torch.zeros(0)
        return x, self.sex[i], t, self.y[i]

_m = BoneAgeNet(replace(CFG, PRETRAINED=False), TDA_X.shape[1])
print(f"{CFG.BACKBONE}: {sum(p.numel() for p in _m.parameters()) / 1e6:.1f} M parameters"); del _m

## 9 · Training engine
AdamW (backbone 1e-4, head 1e-3, wd 0.05) · 1-epoch warm-up + cosine · mixed precision on CUDA · gradient clipping · **EMA** weights for evaluation · early stopping on fold-validation MAE · horizontal-flip TTA at inference. Each `(experiment, fold)` writes `preds.csv`, `history.csv`, `config.json` and `best.pt`; finished runs are loaded instead of retrained.

In [ ]:
def reg_metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return {"MAE_mo": mean_absolute_error(y, p), "RMSE_mo": float(np.sqrt(mean_squared_error(y, p))),
            "R2": r2_score(y, p), "MAE_yr": mean_absolute_error(y, p) / 12}

class EMA:
    def __init__(self, model, decay):
        self.m = copy.deepcopy(model).eval()
        for p in self.m.parameters():
            p.requires_grad_(False)
        self.decay, self.n = decay, 0

    @torch.no_grad()
    def update(self, model):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))
        src = model.state_dict()
        for k, v in self.m.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(src[k].detach(), alpha=1 - d)
            else:
                v.copy_(src[k])

def prepare_fold(cfg, df, tda_all, fold):
    """Train/val/test frames, standardised TDA matrix and the loss, all fitted on training rows only."""
    dev = df[df.split == "dev"]
    tr, va, te = dev[dev.fold != fold], dev[dev.fold == fold], df[df.split == "test"]
    Z = None
    if cfg.USE_TDA:
        L = np.log1p(tda_all)
        mu, sd = L[tr.index].mean(0), L[tr.index].std(0) + 1e-6
        Z = np.clip((L - mu) / sd, -6, 6).astype(np.float32)
    sigma_table, _ = atlas_sigma_table(tr, cfg)
    crit = Criterion(cfg, sigma_table, tr.boneage.mean(), tr.boneage.std()).to(cfg.DEVICE)
    return tr, va, te, Z, crit

def make_loader(frame, Z, cfg, train):
    ds = BoneAgeDataset(frame, None if Z is None else Z[frame.index], train=train)
    return DataLoader(ds, batch_size=cfg.BATCH_SIZE * (1 if train else 2), shuffle=train, drop_last=train,
                      num_workers=cfg.NUM_WORKERS, pin_memory=cfg.DEVICE == "cuda",
                      persistent_workers=cfg.NUM_WORKERS > 0)

def autocast(cfg):
    if cfg.DEVICE == "cuda":
        return torch.autocast("cuda", dtype=torch.float16)
    return contextlib.nullcontext()

@torch.no_grad()
def predict(model, loader, crit, cfg, tta=False):
    model.eval()
    means, sds = [], []
    for x, s, t, _ in loader:
        x, s, t = x.to(cfg.DEVICE, non_blocking=True), s.to(cfg.DEVICE), t.to(cfg.DEVICE)
        mu_v, sd_v = [], []
        for v in ([x, x.flip(-1)] if tta else [x]):
            with autocast(cfg):
                out = model(v, s, t)
            mu, sd = crit.decode(out)
            mu_v.append(mu); sd_v.append(sd)
        mu_v, sd_v = torch.stack(mu_v), torch.stack(sd_v)
        means.append(mu_v.mean(0).cpu())
        sds.append(((sd_v ** 2).mean(0) + mu_v.var(0, unbiased=False)).sqrt().cpu())
    return torch.cat(means).numpy(), torch.cat(sds).numpy()

class OutOfTime(RuntimeError):
    """Raised after checkpointing when the session time budget is used up. Just re-run the cell to resume."""

def _save_atomic(obj, path):
    torch.save(obj, str(path) + ".tmp")
    os.replace(str(path) + ".tmp", path)

def train_run(name, cfg, df, tda_all, fold):
    run_dir = cfg.WORK_DIR / "runs" / name / f"fold{fold}"
    if (run_dir / "preds.csv").exists():
        print(f"[{name} | fold {fold}] found existing run - loading predictions")
        return pd.read_csv(run_dir / "preds.csv")
    run_dir.mkdir(parents=True, exist_ok=True)
    seed_everything(cfg.SEED + fold)
    tr, va, te, Z, crit = prepare_fold(cfg, df, tda_all, fold)
    dl_tr, dl_va, dl_te = make_loader(tr, Z, cfg, True), make_loader(va, Z, cfg, False), make_loader(te, Z, cfg, False)

    model = BoneAgeNet(cfg, 0 if Z is None else Z.shape[1]).to(cfg.DEVICE)
    if cfg.DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)
    head_params = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
    opt = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": cfg.LR_BACKBONE},
                             {"params": head_params, "lr": cfg.LR_HEAD}], weight_decay=cfg.WEIGHT_DECAY)
    total, warm = cfg.EPOCHS * len(dl_tr), max(1, int(cfg.WARMUP_EPOCHS * len(dl_tr)))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda k: (k + 1) / warm if k < warm else 0.5 * (1 + math.cos(math.pi * (k - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.DEVICE == "cuda")
    ema = EMA(model, cfg.EMA_DECAY)

    best, bad, history, start_epoch = float("inf"), 0, [], 0
    last = run_dir / "last.pt"
    if last.exists():      # resume an interrupted run from the last finished epoch
        ck = torch.load(last, map_location=cfg.DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"]); ema.m.load_state_dict(ck["ema"]); ema.n = ck["ema_n"]
        best, bad, history, start_epoch = ck["best"], ck["bad"], ck["history"], ck["epoch"]
        print(f"[{name} | fold {fold}] resuming from epoch {start_epoch} (best val MAE {best:.2f} mo)")
    for epoch in range(start_epoch, cfg.EPOCHS):
        if bad >= cfg.PATIENCE:
            break
        model.train()
        t0, run_loss = time.time(), 0.0
        for x, s, t, y in tqdm(dl_tr, desc=f"{name} f{fold} e{epoch + 1}", leave=False):
            x, s, t, y = (a.to(cfg.DEVICE, non_blocking=True) for a in (x, s, t, y))
            if cfg.DEVICE == "cuda":
                x = x.contiguous(memory_format=torch.channels_last)
            with autocast(cfg):
                out = model(x, s, t)
            loss = crit(out, y, s)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(opt); scaler.update(); sched.step()
            ema.update(model)
            run_loss += loss.item()
        vmu, _ = predict(ema.m, dl_va, crit, cfg)
        m = reg_metrics(va.boneage, vmu)
        history.append({"epoch": epoch + 1, "loss": run_loss / len(dl_tr), **m, "sec": time.time() - t0})
        flag = ""
        if m["MAE_mo"] < best:
            best, bad, flag = m["MAE_mo"], 0, " *"
            torch.save(ema.m.state_dict(), run_dir / "best.pt")
        else:
            bad += 1
        print(f"[{name} | fold {fold}] epoch {epoch + 1:02d}  loss {history[-1]['loss']:.4f}  "
              f"val MAE {m['MAE_mo']:.2f} mo  R² {m['R2']:.4f}  ({history[-1]['sec']:.0f}s){flag}")
        _save_atomic({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                      "scaler": scaler.state_dict(), "ema": ema.m.state_dict(), "ema_n": ema.n, "best": best,
                      "bad": bad, "history": history, "epoch": epoch + 1}, last)
        if bad >= cfg.PATIENCE:
            break
        if cfg.TIME_BUDGET_MIN and (time.time() - SESSION_T0) / 60 > cfg.TIME_BUDGET_MIN:
            raise OutOfTime(f"time budget reached after epoch {epoch + 1} of [{name} | fold {fold}]; "
                            f"checkpoint saved - download/commit {cfg.WORK_DIR} and re-run this cell to resume.")

    ema.m.load_state_dict(torch.load(run_dir / "best.pt", map_location=cfg.DEVICE))
    vmu, vsd = predict(ema.m, dl_va, crit, cfg, tta=cfg.TTA)
    tmu, tsd = predict(ema.m, dl_te, crit, cfg, tta=cfg.TTA)
    cols = ["id", "boneage", "male"]
    preds = pd.concat([va[cols].assign(part="val", pred=vmu, pred_sd=vsd),
                       te[cols].assign(part="test", pred=tmu, pred_sd=tsd)], ignore_index=True)
    preds["pred"] = preds["pred"].clip(0, cfg.MAX_AGE)
    preds.to_csv(run_dir / "preds.csv", index=False)
    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
    (run_dir / "config.json").write_text(json.dumps(asdict(cfg), default=str, indent=1))
    last.unlink(missing_ok=True)
    del model, ema, opt
    gc.collect()
    if cfg.DEVICE == "cuda":
        torch.cuda.empty_cache()
    return preds

## 10 · Experiments

### 10.1 Ablation (fold 0, single model + flip TTA, scored on the held-out test set)
A 2 × 2 factorial over {loss: L1 / AADL} × {topology stream: off / on}, plus the Laplace-NLL loss as a second baseline. Every run shares the same preprocessing, backbone, schedule and split, so the differences isolate the contribution of each idea.

In [ ]:
EXPERIMENTS = {
    "E0_l1":        dict(LOSS="l1",      USE_TDA=False),   # modernised baseline
    "E1_laplace":   dict(LOSS="laplace", USE_TDA=False),
    "E2_aadl":      dict(LOSS="aadl",    USE_TDA=False),
    "E3_l1_tda":    dict(LOSS="l1",      USE_TDA=True),
    "E4_aadl_tda":  dict(LOSS="aadl",    USE_TDA=True),    # full model
}
ABLATION_RUNS = list(EXPERIMENTS)       # trim this list to save compute
ABLATION_FOLD = 0
FINAL_EXP = "E4_aadl_tda"               # switch if the ablation picks a different winner
FINAL_FOLDS = list(range(CFG.N_FOLDS))

ablation = {name: train_run(name, replace(CFG, **EXPERIMENTS[name]), df, TDA_X, ABLATION_FOLD)
            for name in ABLATION_RUNS}

In [ ]:
rows = []
for name, p in ablation.items():
    te = p[p.part == "test"]
    rows.append({"experiment": name, **EXPERIMENTS[name], **reg_metrics(te.boneage, te.pred)})
abl = pd.DataFrame(rows).set_index("experiment")
display(abl.style.format({"MAE_mo": "{:.2f}", "RMSE_mo": "{:.2f}", "R2": "{:.4f}", "MAE_yr": "{:.3f}"})
        .highlight_min(subset=["MAE_mo", "RMSE_mo"], color="#cde7d8").highlight_max(subset=["R2"], color="#cde7d8"))

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].barh(abl.index, abl.MAE_mo, color="#2e86ab"); ax[0].set(xlabel="test MAE (months)", title="Ablation — MAE ↓")
ax[1].barh(abl.index, abl.R2, color="#3b8f6a"); ax[1].set(xlabel="test R²", title="Ablation — R² ↑")
ax[1].set_xlim(max(0.0, abl.R2.min() - 0.02), 1.0)
for a in ax: a.invert_yaxis()
plt.tight_layout(); plt.savefig(CFG.WORK_DIR / "figures/ablation.png", dpi=130); plt.show()

### 10.2 Final model — 5-fold ensemble
The chosen configuration is trained on all folds (fold 0 is reused from the ablation). Each fold model predicts its own validation fold (**out-of-fold**, used for calibration) and the full test set (averaged into the ensemble).

In [ ]:
final_cfg = replace(CFG, **EXPERIMENTS[FINAL_EXP])
fold_preds = [train_run(FINAL_EXP, final_cfg, df, TDA_X, f) for f in FINAL_FOLDS]

oof = pd.concat([p[p.part == "val"] for p in fold_preds], ignore_index=True)
stack = pd.concat([p[p.part == "test"] for p in fold_preds], ignore_index=True)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    ens = stack.groupby("id").agg(boneage=("boneage", "first"), male=("male", "first"), pred=("pred", "mean"),
                                  var_within=("pred_sd", lambda s: np.nanmean(s ** 2)),
                                  var_between=("pred", lambda s: s.var(ddof=0))).reset_index()
ens["pred_sd"] = np.sqrt(ens.var_within + ens.var_between)
single = fold_preds[0][fold_preds[0].part == "test"]
print(f"OOF ({len(oof)} imgs):            ", {k: round(v, 4) for k, v in reg_metrics(oof.boneage, oof.pred).items()})
print(f"Test, single fold-0 model:      ", {k: round(v, 4) for k, v in reg_metrics(single.boneage, single.pred).items()})
print(f"Test, {len(fold_preds)}-fold ensemble:        ", {k: round(v, 4) for k, v in reg_metrics(ens.boneage, ens.pred).items()})

## 11 · Post-processing

**Regression-to-the-mean calibration.** Fit, per sex, a linear map *predicted → true* on the out-of-fold predictions. Whether to apply it is decided by 5-fold cross-validation **inside the OOF set** — the test set plays no part in the decision.

**Split-conformal intervals.** With normalised scores $|y-\hat y|/\hat\sigma$ on the OOF set, the $\lceil (n+1)(1-\alpha)\rceil / n$ quantile $\hat q$ gives intervals $\hat y \pm \hat q\,\hat\sigma$ with ≥ 90 % marginal coverage (exchangeability assumed). With the plain-L1 model, $\hat\sigma \equiv 1$ (constant-width intervals).

In [ ]:
def fit_calibrator(frame):
    return {s: LinearRegression().fit(g[["pred"]], g["boneage"]) for s, g in frame.groupby("male")}

def apply_calibrator(cal, frame):
    out = frame["pred"].to_numpy(dtype=float).copy()
    for s, m in cal.items():
        idx = (frame["male"] == s).to_numpy()
        if idx.any():
            out[idx] = m.predict(frame.loc[idx, ["pred"]])
    return np.clip(out, 0, CFG.MAX_AGE)

cv_pred = np.zeros(len(oof))
for tr_i, va_i in KFold(5, shuffle=True, random_state=CFG.SEED).split(oof):
    cv_pred[va_i] = apply_calibrator(fit_calibrator(oof.iloc[tr_i]), oof.iloc[va_i])
mae_raw, mae_cal = mean_absolute_error(oof.boneage, oof.pred), mean_absolute_error(oof.boneage, cv_pred)
USE_CAL = mae_cal < mae_raw
print(f"OOF MAE raw {mae_raw:.3f} -> cross-validated calibrated {mae_cal:.3f}  =>  apply calibration: {USE_CAL}")

calibrator = fit_calibrator(oof)
oof["pred_final"] = apply_calibrator(calibrator, oof) if USE_CAL else oof["pred"]
ens["pred_final"] = apply_calibrator(calibrator, ens) if USE_CAL else ens["pred"]
for s, m in calibrator.items():
    print(f"  {SEX_NAMES[s]:>6}: true ≈ {m.coef_[0]:.3f}·pred {m.intercept_:+.2f}")

# split-conformal
has_sd = oof["pred_sd"].notna().all() and ens["pred_sd"].notna().all()
scale_oof = oof["pred_sd"].to_numpy() if has_sd else np.ones(len(oof))
scale_te = ens["pred_sd"].to_numpy() if has_sd else np.ones(len(ens))
scores = np.abs(oof.boneage - oof.pred_final).to_numpy() / scale_oof
n = len(scores)
q_hat = np.quantile(scores, min(1.0, math.ceil((n + 1) * (1 - CFG.CONFORMAL_ALPHA)) / n), method="higher")
ens["lo"], ens["hi"] = ens.pred_final - q_hat * scale_te, ens.pred_final + q_hat * scale_te
coverage = ((ens.boneage >= ens.lo) & (ens.boneage <= ens.hi)).mean()
print(f"Conformal {1 - CFG.CONFORMAL_ALPHA:.0%} intervals on test: coverage {coverage:.1%}, "
      f"mean width {(ens.hi - ens.lo).mean():.1f} mo ({'adaptive' if has_sd else 'constant'} width)")

## 12 · Final evaluation on the held-out test set

In [ ]:
FINAL = reg_metrics(ens.boneage, ens.pred_final)
per_sex = pd.DataFrame({SEX_NAMES[s]: reg_metrics(g.boneage, g.pred_final) for s, g in ens.groupby("male")}).T
display(Markdown(f"**Test set (n = {len(ens)})** — MAE **{FINAL['MAE_mo']:.2f} mo** ({FINAL['MAE_yr']:.3f} y) · "
                 f"RMSE **{FINAL['RMSE_mo']:.2f} mo** · R² **{FINAL['R2']:.4f}**"))
display(per_sex.round(4))

y, p = ens.boneage.to_numpy(), ens.pred_final.to_numpy()
fig, ax = plt.subplots(2, 2, figsize=(14, 12))
for s in (0, 1):
    g = ens[ens.male == s]
    ax[0, 0].scatter(g.boneage / 12, g.pred_final / 12, s=10, alpha=0.45, color=SEX_COLORS[s], label=SEX_NAMES[s])
ax[0, 0].plot([0, 20], [0, 20], "k--", lw=1)
ax[0, 0].set(xlabel="True bone age (y)", ylabel="Predicted (y)",
             title=f"Predicted vs true — R² {FINAL['R2']:.4f}, MAE {FINAL['MAE_mo']:.2f} mo"); ax[0, 0].legend()

mean_, diff_ = (y + p) / 2 / 12, (p - y) / 12
bias, loa = diff_.mean(), 1.96 * diff_.std()
ax[0, 1].scatter(mean_, diff_, s=8, alpha=0.4, color="#555")
for v, ls in ((bias, "-"), (bias + loa, "--"), (bias - loa, "--")):
    ax[0, 1].axhline(v, color="#d1495b", ls=ls, lw=1.2)
ax[0, 1].set(xlabel="Mean of true & predicted (y)", ylabel="Predicted − true (y)",
             title=f"Bland–Altman — bias {bias*12:+.2f} mo, 95 % LoA ±{loa*12:.1f} mo")

bins = np.arange(0, 21, 2)
ens["age_band"] = pd.cut(ens.boneage / 12, bins, right=False)
band = ens.assign(ae=(ens.pred_final - ens.boneage).abs()).groupby(["age_band", "male"], observed=True)["ae"].mean().unstack()
band.rename(columns=SEX_NAMES).plot.bar(ax=ax[1, 0], color=[SEX_COLORS[0], SEX_COLORS[1]], width=0.8)
ax[1, 0].set(xlabel="True age band (y)", ylabel="MAE (months)", title="Error by age and sex"); ax[1, 0].legend(title="")

if has_sd:
    ens["sd_q"] = pd.qcut(ens.pred_sd, 10, labels=False, duplicates="drop")
    unc = ens.assign(ae=(ens.pred_final - ens.boneage).abs()).groupby("sd_q").agg(sd=("pred_sd", "mean"), ae=("ae", "mean"))
    ax[1, 1].plot(unc.sd, unc.ae, "o-", color="#3b8f6a", lw=2)
    lim = [0, max(unc.sd.max(), unc.ae.max()) * 1.1]
    ax[1, 1].plot(lim, [l * math.sqrt(2 / math.pi) for l in lim], "k--", lw=1, label="ideal for Gaussian errors")
    ax[1, 1].set(xlabel="Predicted σ (months, decile mean)", ylabel="Observed MAE (months)",
                 title=f"Is the uncertainty informative?  conformal 90 % coverage {coverage:.1%}"); ax[1, 1].legend()
else:
    ax[1, 1].axis("off"); ax[1, 1].text(0.5, 0.5, "no predictive σ for this loss", ha="center")
plt.tight_layout(); plt.savefig(CFG.WORK_DIR / "figures/final_evaluation.png", dpi=150); plt.show()

**Developmental stage** (derived from the regression output; one consistent definition: 0–3 / 3–10 / 10–15 / 15+ years).

In [ ]:
STAGE_EDGES = [36, 120, 180]
STAGE_NAMES = ["Toddler 0–3y", "Pre-puberty 3–10y", "Puberty 10–15y", "Young adult 15y+"]
st_true, st_pred = np.digitize(ens.boneage, STAGE_EDGES), np.digitize(ens.pred_final, STAGE_EDGES)
STAGE = {"accuracy": accuracy_score(st_true, st_pred), "macro_F1": f1_score(st_true, st_pred, average="macro"),
         "QWK": cohen_kappa_score(st_true, st_pred, weights="quadratic")}
print({k: round(v, 4) for k, v in STAGE.items()})
cm = confusion_matrix(st_true, st_pred, labels=range(4))
plt.figure(figsize=(7, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=STAGE_NAMES, yticklabels=STAGE_NAMES, cbar=False)
plt.xlabel("Predicted stage"); plt.ylabel("True stage"); plt.title("Developmental stage — test set")
plt.tight_layout(); plt.savefig(CFG.WORK_DIR / "figures/stage_confusion.png", dpi=130); plt.show()

## 13 · Interpretability — Grad-CAM on the predicted age
Gradients of the predicted age (the expectation of the AADL distribution) with respect to the last backbone feature map.

In [ ]:
def gradcam(model, x, s, t, crit):
    model.eval()
    with torch.enable_grad():
        fmap = model.backbone.forward_features(x)
        fmap.retain_grad()
        mu, _ = crit.decode(model.head_from_features(fmap, s, t))
        model.zero_grad(); mu.sum().backward()
    w = fmap.grad.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * fmap).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
    return (cam / (cam.max() + 1e-8)).detach().cpu().numpy(), float(mu.item())

fold = FINAL_FOLDS[0]
tr_, va_, te_, Z_, crit_ = prepare_fold(final_cfg, df, TDA_X, fold)
cam_model = BoneAgeNet(replace(final_cfg, PRETRAINED=False), 0 if Z_ is None else Z_.shape[1]).to(final_cfg.DEVICE)
cam_model.load_state_dict(torch.load(final_cfg.WORK_DIR / "runs" / FINAL_EXP / f"fold{fold}" / "best.pt",
                                     map_location=final_cfg.DEVICE))
ds_ = BoneAgeDataset(te_, None if Z_ is None else Z_[te_.index])
picks = [int(np.abs(te_.boneage.to_numpy() - a).argmin()) for a in (18, 72, 132, 192)]
fig, ax = plt.subplots(1, 4, figsize=(18, 5))
for a, i in zip(ax, picks):
    x, s, t, y = ds_[i]
    cam, mu = gradcam(cam_model, x[None].to(final_cfg.DEVICE), torch.tensor([s], device=final_cfg.DEVICE),
                      t[None].to(final_cfg.DEVICE), crit_)
    a.imshow(x[0].numpy(), cmap="gray"); a.imshow(cam, cmap="jet", alpha=0.35); a.axis("off")
    a.set_title(f"true {y/12:.1f} y · pred {mu/12:.1f} y")
plt.tight_layout(); plt.savefig(CFG.WORK_DIR / "figures/gradcam.png", dpi=130); plt.show()
del cam_model

## 14 · Results summary

In [ ]:
SUBMISSION = {"MAE_mo": 7.95, "RMSE_mo": 10.79, "R2": 0.9343}
abl_md = "\n".join(f"| {k} | {r.LOSS} | {'✓' if r.USE_TDA else ''} | {r.MAE_mo:.2f} | {r.RMSE_mo:.2f} | {r.R2:.4f} |"
                   for k, r in abl.iterrows())
report = f"""
| Model | Evaluation set | MAE (mo) | RMSE (mo) | R² |
|---|---|---|---|---|
| Submission — Xception + ResNet50 | validation (also used for checkpoint selection) | {SUBMISSION['MAE_mo']:.2f} | {SUBMISSION['RMSE_mo']:.2f} | {SUBMISSION['R2']:.4f} |
| Topology-only gradient boosting | held-out test | {TDA_ONLY['MAE_mo']:.2f} | – | {TDA_ONLY['R2']:.4f} |
| **Iteration 1 — {FINAL_EXP}, {len(FINAL_FOLDS)}-fold ensemble{' + calibration' if USE_CAL else ''}** | **held-out test** | **{FINAL['MAE_mo']:.2f}** | **{FINAL['RMSE_mo']:.2f}** | **{FINAL['R2']:.4f}** |

**Ablation (fold-0 single model, held-out test)**

| Experiment | Loss | Topology | MAE (mo) | RMSE (mo) | R² |
|---|---|---|---|---|---|
{abl_md}

Stage classification: accuracy {STAGE['accuracy']:.3f}, macro-F1 {STAGE['macro_F1']:.3f}, QWK {STAGE['QWK']:.3f}.
Conformal 90 % intervals: coverage {coverage:.1%}, mean width {(ens.hi - ens.lo).mean():.1f} mo.
"""
display(Markdown(report))
results = {"final": FINAL, "per_sex": per_sex.to_dict(), "ablation": abl.drop(columns=["LOSS", "USE_TDA"]).to_dict("index"),
           "tda_only": TDA_ONLY, "stage": STAGE, "conformal": {"coverage": float(coverage), "q_hat": float(q_hat)},
           "calibration_applied": bool(USE_CAL), "config": asdict(final_cfg)}
(CFG.WORK_DIR / "results.json").write_text(json.dumps(results, default=str, indent=1))
(CFG.WORK_DIR / "results.md").write_text(report)
print(f"saved results and figures to {CFG.WORK_DIR}")

## 15 · Next iteration
* **Attention-zoom stream** — use the Grad-CAM peaks of this model to crop high-resolution patches (physes, carpals) for a second-stage network. This replaces the submission's fixed crops with learned ones.
* **Resolution** — 640–768 px inputs or a larger backbone (ConvNeXt-S, EfficientNetV2-M) for the final ensemble.
* **External validation** — the official RSNA 2017 test set (200 images with consensus labels, via Stanford AIMI) to compare directly against the challenge leaderboard (winner MAE ≈ 4.3 mo).
* **Topology ablations** — persistence images vs. Betti curves; learned filtrations.